# Первый эксперимент: смысл контракта и поиск похожих пар

29 сентября 2026. **Выполненный учебный notebook, не benchmark качества NLP.**
Данные: сохранённые публичные тексты Kalshi. Интернет, аккаунт и LLM API не нужны.
Необходимы библиотеки из `pilot/requirements.txt`. Для нового запуска выберите Python 3.11.

Цель: увидеть, почему сходство текста не равно логической связи, и научиться
различать эмпирический пилот и синтетическую проверку метрики.


In [1]:
from pathlib import Path
import json, subprocess, sys
root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
assert (root / 'pilot/markets.json').exists(), 'Откройте notebook из корня проекта или notebooks/'
markets = json.loads((root / 'pilot/markets.json').read_text())
print('Сохранённых контрактов:', len(markets))
print('Площадки:', sorted({m['venue'] for m in markets}))


Сохранённых контрактов: 286
Площадки: ['kalshi']


## 1. Пересчитать retrieval и проверить целостность данных

Скрипт сверяет SHA256 исходных JSON, нормализованного снимка и входов.
Затем считает три TF-IDF представления. Он использует квадратичную матрицу;
это небольшая демонстрация, не реализация масштабного ANN-поиска.
Вывод ниже получен исполнением кода, не заполнен вручную.


In [2]:
proc = subprocess.run([sys.executable, str(root/'pilot/pilot.py'), 'analyze', '--top-k', '5'],
                      check=True, capture_output=True, text=True, cwd=root)
print(proc.stdout)


{
  "generated_at_utc": "2026-09-28T21:06:59.527295+00:00",
  "input_markets_sha256": "b563a9b5948e5c7a485db0919d54739afdbed5e6e7cd491620db4a69d7dada97",
  "python": "3.11.4",
  "dependencies": {
    "requests": "2.31.0",
    "numpy": "1.25.1",
    "scikit-learn": "1.3.0"
  },
  "markets": 286,
  "venues": {
    "kalshi": 286
  },
  "events": 19,
  "missing_primary": 0,
  "missing_secondary": 46,
  "identical_title_groups": 1,
  "possible_pairs": 40755,
  "top_k": 5,
  "candidate_union_pairs": 1188,
  "union_fraction_of_all_pairs": 0.029149797570850202,
  "retrievers": {
    "title": {
      "candidate_pairs": 888,
      "cross_event_pairs": 346
    },
    "rules": {
      "candidate_pairs": 943,
      "cross_event_pairs": 147
    },
    "title_rules": {
      "candidate_pairs": 953,
      "cross_event_pairs": 186
    }
  },
  "interpretation": "Engineering and descriptive results only. No gold labels, accuracy, recall or tradable arbitrage measurements."
}



**Как читать результат:** 1 188 — число отобранных пар. 2,915% — доля от 40 755
возможных пар. У нас ещё нет gold-разметки, поэтому recall/precision неизвестны.
Выборка нерепрезентативна; Polymarket не был доступен при сборе.


In [3]:
proc = subprocess.run([sys.executable, str(root/'pilot/review_cases.py')],
                      check=True, capture_output=True, text=True, cwd=root)
cases = json.loads((root/'pilot/results/review_cases.json').read_text())
print('Подготовленных разборов:', len(cases))
by_case = {c['case_id']: c for c in cases}
case = by_case['same_title_thresholds']
for side in ['a','b']:
    m = case[side]
    print(side.upper(), m['id'], '\nЗаголовок:', m['title'], '\nПравило:', m['primary'], '\n')
print('Сходство заголовков:', case['retrieval']['title']['similarity'])


Подготовленных разборов: 6
A KXLCPIMAXYOY-27-P7.0 
Заголовок: Inflation surge in 2026? 
Правило: If year-over-year CPI inflation is at least 7.0% in any month in 2026, then the market resolves to Yes. 

B KXLCPIMAXYOY-27-P6.5 
Заголовок: Inflation surge in 2026? 
Правило: If year-over-year CPI inflation is at least 6.5% in any month in 2026, then the market resolves to Yes. 

Сходство заголовков: 1.0


## 2. Ваше рассуждение

Одинаковые заголовки скрывают пороги ≥7% и ≥6,5%. При одинаковых остальных
условиях первый влечёт второй, но не наоборот. Попробуйте придумать число,
при котором второй Yes, а первый No. Затем проверьте полные условия серии:
API primary rules ещё не гарантируют полноту спецификации.
Интерпретации кейсов сделаны с помощью AI; это не независимый человеческий gold.


In [4]:
case = by_case['different_periods']
print(case['a']['primary'])
print(case['b']['primary'])
print('Показатели поиска:', case['retrieval']['rules'])
print('Предварительный разбор:', case['note_ru'])


If the Consumer Price Index (CPI) for All Urban Consumers: All Items less Food and Energy increases by more than 2.6% in the twelve months ending December 2026 (as represented by the one-decimal place value reported by the Bureau of Labor Statistics), then the market resolves to Yes.
If the Consumer Price Index (CPI) for All Urban Consumers: All Items less Food and Energy increases by more than 2.6% in the twelve months ending November 2026 (as represented by the one-decimal place value reported by the Bureau of Labor Statistics), then the market resolves to Yes.
Показатели поиска: {'similarity': 0.973437, 'rank_b_for_a': 1, 'rank_a_for_b': 1, 'in_top5_union': True}
Предварительный разбор: Различаются двенадцатимесячные периоды: заканчивающийся в декабре и в ноябре. Высокая близость текстов не доказывает эквивалентность или импликацию. Это не утверждение статистической независимости.


Почти одинаковый текст и rank=1 не доказывают следование: месяцы различаются.
И наоборот, полезная пороговая пара может оказаться за пределами top-5.


In [5]:
case = by_case['strict_thresholds']
print('A:', case['a']['title'])
print('B:', case['b']['title'])
for view, score in case['retrieval'].items():
    print(view, score)
assert all(not v['in_top5_union'] for v in case['retrieval'].values())


A: Will CPI rise more than 0.9% in October 2026?
B: Will CPI rise more than 0.8% in October 2026?
title {'similarity': 0.674246, 'rank_b_for_a': 14, 'rank_a_for_b': 14, 'in_top5_union': False}
rules {'similarity': 0.946437, 'rank_b_for_a': 7, 'rank_a_for_b': 8, 'in_top5_union': False}
title_rules {'similarity': 0.894755, 'rank_b_for_a': 7, 'rank_a_for_b': 9, 'in_top5_union': False}


## 3. Проверить метрики без LLM

Следующие числа получены на **искусственно заданных мирах**, а не по рыночной
выборке. Они проверяют корректность определения метрики.


In [6]:
sys.path.insert(0, str(root/'logic'))
from metric_demo import run
demo = run()
print(json.dumps(demo, ensure_ascii=False, indent=2))


{
  "kind": "synthetic_exact_enumeration_not_empirical_benchmark",
  "implication": {
    "definition": "A: x>150; B: x>100; same source/time; binary, no exceptions",
    "feasible_states": [
      [
        0,
        0
      ],
      [
        0,
        1
      ],
      [
        1,
        1
      ]
    ],
    "labels": [
      "a_implies_b"
    ],
    "counterexample_to_equality": [
      0,
      1
    ],
    "positive_sign_equality_proxy_uniform_worlds": 0.6666666666666666,
    "implication_violation_rate": 0.0
  },
  "exclusive_triangle": {
    "stipulated_worlds": [
      [
        1,
        0,
        0
      ],
      [
        0,
        1,
        0
      ],
      [
        0,
        0,
        1
      ]
    ],
    "exclusion_violations": 0,
    "minimum_signed_frustrated_edges": 1,
    "signed_frustration_rate": 0.3333333333333333,
    "opposite_outcome_proxy_uniform_worlds": 0.6666666666666666,
    "both_no_witness_for_first_pair": [
      0,
      0,
      1
    ]
  },

Треугольник из трёх взаимоисключающих победителей логически корректен,
но его signed frustration равна 1/3. Два независимых события с P(Yes)=0,1
совпадают в среднем в 82% случаев. Поэтому нужны метрики конкретных
ограничений выплат и независимая разметка.

## 4. Первый самостоятельный шаг

Выберите ещё одну пару в `pilot/results/candidates.json`. Найдите её контракты
в `markets.json`. Выпишите объект измерения, период, источник, порог, исключения.
Для каждого состояния 00/01/10/11 укажите: запрещено с основанием,
возможно с конкретным сценарием, или неизвестно. Не смешивайте unknown
с доказанным отсутствием логической связи.

Дальше сравнивать NLI/LLM можно только после согласования инструкции
`protocol/annotation_guide.md` и получения gold. Этот notebook не запускает
NLI/LLM, не совершает сделок и не воспроизводит численные результаты статьи.
